# 🤖 Model Training & Evaluation

In this notebook, we will train 3 different machine learning models to predict customer churn:
1. **Logistic Regression** (Baseline)
2. **Random Forest** (Ensemble)
3. **XGBoost** (Advanced Gradient Boosting)

We will compare their performance, focusing especially on **Recall** (ability to correctly identify churning customers). Finally, we'll save the best model and explain it using **SHAP** values.

In [ ]:
import sys
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

from src.feature_engineering import create_features
from src.preprocessing import preprocess_pipeline, split_data
from src.model_training import train_and_evaluate, select_best_model, save_model, save_results_report

# Load and prepare data (same as previous notebook)
df = pd.read_csv("../data/raw/telco_customer_churn.csv")
df_featured = create_features(df)
df_processed = preprocess_pipeline(df_featured, encode_method='onehot', scale=True, fit_scaler=True)
X_train, X_test, y_train, y_test = split_data(df_processed)

## 1. Train Models
We will train the 3 models and evaluate their performance on the test set.

In [ ]:
results = train_and_evaluate(X_train, X_test, y_train, y_test)

# Compare results in a DataFrame
comparison_df = pd.DataFrame({
    name: {k: v for k, v in metrics.items() if k != 'model_instance' and k != 'confusion_matrix'}
    for name, metrics in results.items()
}).T

display(comparison_df.sort_values(by='recall', ascending=False))

## 2. Select Best Model & Save
We choose the model with the highest **Recall**, because in Churn Prediction, false negatives (missing a customer who will churn) are much more costly than false positives (giving a retention offer to a loyal customer).

In [ ]:
# Select best model based on Recall
best_name, best_model, best_score = select_best_model(results, metric="recall")

# Save model & report
save_model(best_model, best_name, "../models/churn_model.pkl")
save_results_report(results, "../models/training_report.json")

print("\nSaved Model and Report successfully!")

## 3. SHAP Model Explainability
To explain the model to business stakeholders, we use **SHAP** values. This tells us *why* the model makes a certain prediction.

In [ ]:
import shap
import matplotlib.pyplot as plt
# We use Logistic Regression for simple, fast SHAP explanations
explainer = shap.LinearExplainer(results['Logistic Regression']['model_instance'], X_train)
shap_values = explainer.shap_values(X_test)

plt.title("SHAP Summary Plot (Feature Importance & Impact)", pad=20)
shap.summary_plot(shap_values, X_test, feature_names=X_test.columns)